# 数据输入、质量控制与主连复权

本 Notebook 通过首个配置单元在 `excel` 与 `duckdb` 两种数据源之间切换。两条入口在读取后进入同一套字段标准化、质量控制、主连复权和 Parquet 输出流程，供移仓监控、信号生成和回测直接读取。


In [1]:
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for candidate in _PROJECT_CANDIDATES if (candidate / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("找不到项目根目录 pyproject.toml；请从本项目内启动 Jupyter。")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import duckdb
import pandas as pd

from factors.paths import (
    DB_PATH as DEFAULT_DB_PATH,
    FACTOR_WORKING_DIR,
    INPUT_DIR as DEFAULT_INPUT_DIR,
)

PACKAGE_DIR = PROJECT_ROOT
INPUT_DIR = DEFAULT_INPUT_DIR
WORKING_DIR = FACTOR_WORKING_DIR

######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====
DATA_SOURCE = "duckdb"  # 二选一："excel" 读取下方文件；"duckdb" 读取 TreasuryFutures.ipynb 生成的数据库
######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====

######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====
DUCKDB_PATH = DEFAULT_DB_PATH
RAW_DAILY_PATH = INPUT_DIR / "T_daily.parquet"  # 原始日线：xlsx/xls/parquet/csv 均可
MAIN_MAPPING_PATH = INPUT_DIR / "main_mapping.parquet"  # 主力映射；手工日线必须提供
EXTERNAL_DAILY_PATH = INPUT_DIR / "external_daily.parquet"  # 外部连续复权日线
RAW_MINUTE_PATH = INPUT_DIR / "T_minute.csv"  # 原始分钟线
EXTERNAL_MINUTE_PATH = INPUT_DIR / "external_minute.parquet"  # 外部连续复权分钟线
######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====

######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====
EVENT_CALENDAR_PATH = INPUT_DIR / "eco_calendar_filtered.xlsx"  # 两种数据源共用的宏观事件日历
LEGACY_EVENT_CALENDAR_PATH = INPUT_DIR / "event_calendar.parquet"  # 兼容旧版测试/输入
######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====######=====

MANUAL_ADJUST_DAILY = True  # True 按主力映射手工计算后复权；False 使用外部复权日线
MANUAL_ADJUST_MINUTE = True  # True 套用日线因子；False 使用外部复权分钟线
ADJUST_METHOD = "backward_ratio"  # 比例后复权：最新交易日因子为 1，历史价格乘随后换月比例
ROLL_PRICE_COLUMN = "settle"  # 换月比例优先使用前一交易日结算价；没有时可改 close

RAW_DAILY_MAP = {"contract": "contract", "trade_date": "trade_date", "open": "open", "high": "high", "low": "low", "close": "close", "settle": "settle", "oi": "oi", "amount": "amount", "volume": "volume"}  # 原始日线 source→标准名
MAIN_MAPPING_MAP = {"trade_date": "trade_date", "main_contract": "main_contract"}  # 映射 source→标准名
EXTERNAL_DAILY_MAP = RAW_DAILY_MAP  # 外部日线必须含 contract 或 main_contract
RAW_MINUTE_MAP = {"datetime": "datetime", "trading_date": "trading_date", "contract": "contract", "open": "open", "high": "high", "low": "low", "close": "close", "oi": "oi", "amount": "amount", "volume": "volume"}  # 原始分钟 source→标准名
EXTERNAL_MINUTE_MAP = RAW_MINUTE_MAP  # 外部分钟 source→标准名
EVENT_CALENDAR_MAP = {"date": "event_date", "indicator": "event_name", "tf_category": "event_type", "event_date": "event_date", "event_name": "event_name", "event_type": "event_type"}  # eco_calendar_filtered 与旧版事件表兼容


## 输入约定与选择

只需修改 `DATA_SOURCE`：`"excel"` 使用显式文件路径（也兼容 xls/parquet/csv），`"duckdb"` 使用 `data/database/treasury_futures.duckdb`。Excel 模式下，原始日线需要合约、交易日、OHLC、持仓、成交额和成交量；当 `ROLL_PRICE_COLUMN="settle"` 时还必须映射结算价。两种模式共用事件日历、复权和最终输出。


In [2]:
def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"不支持的文件格式: {path}")


def canonicalize_frame(frame: pd.DataFrame, mapping: dict[str, str], label: str, required: list[str]) -> pd.DataFrame:
    frame = frame.rename(columns=mapping).copy()  # 显式映射后才进入标准流程
    missing = [column for column in required if column not in frame.columns]
    if missing:
        detail = "；ROLL_PRICE_COLUMN=settle 时请映射 settle，或由用户显式改为 close" if "settle" in missing else ""
        raise ValueError(f"{label} 缺少必需列: {missing}{detail}")
    return frame


def canonicalize(path: Path, mapping: dict[str, str], label: str, required: list[str]) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"{label} 文件不存在: {path}")
    return canonicalize_frame(read_table(path), mapping, label, required)


def load_duckdb_sources(path: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not path.is_file():
        raise FileNotFoundError(f"DuckDB 文件不存在: {path}")
    con = duckdb.connect(str(path), read_only=True)
    try:
        available = {row[0] for row in con.execute("SHOW TABLES").fetchall()}
        required_tables = {"daily_bars", "main_contract_mapping", "main_daily_continuous", "main_minute_continuous"}
        missing = sorted(required_tables - available)
        if missing:
            raise ValueError(f"DuckDB 缺少 TreasuryFutures.ipynb 应生成的表: {missing}")
        raw_daily_db = con.execute(
            """
            SELECT trade_date, wind_code AS contract, open, high, low, close, settle,
                   oi, amt AS amount, volume
            FROM daily_bars
            ORDER BY trade_date, wind_code
            """
        ).fetchdf()
        daily_main_db = con.execute(
            """
            SELECT trade_date, main_contract AS contract, open, high, low, close, settle,
                   oi, amt AS amount, volume
            FROM main_daily_continuous
            ORDER BY trade_date
            """
        ).fetchdf()
        minute_main_db = con.execute(
            """
            SELECT bar_time AS datetime, trade_date AS trading_date,
                   main_contract AS contract, open, high, low, close,
                   oi, amt AS amount, volume
            FROM main_minute_continuous
            ORDER BY bar_time
            """
        ).fetchdf()
        mapping_db = con.execute(
            """
            SELECT trade_date, main_contract
            FROM main_contract_mapping
            ORDER BY trade_date
            """
        ).fetchdf()
    finally:
        con.close()

    complete_columns = ["open", "high", "low", "close", ROLL_PRICE_COLUMN, "oi", "amount", "volume"]
    raw_complete = raw_daily_db[complete_columns].notna().all(axis=1)
    if (~raw_complete).any():
        print(f"DuckDB 原始日线剔除 {int((~raw_complete).sum()):,} 行不完整辅助合约记录；这些记录不进入主连或换月价格计算。")
    raw_daily_db = raw_daily_db.loc[raw_complete].reset_index(drop=True)

    main_complete = daily_main_db[complete_columns].notna().all(axis=1)
    if (~main_complete).any():
        omitted_dates = pd.to_datetime(daily_main_db.loc[~main_complete, "trade_date"]).dt.strftime("%Y-%m-%d").tolist()
        print(f"DuckDB 主连日线剔除 {len(omitted_dates)} 个尚未完整的交易日: {omitted_dates}")
    daily_main_db = daily_main_db.loc[main_complete].reset_index(drop=True)
    return raw_daily_db, daily_main_db, minute_main_db, mapping_db


def parse_and_validate(frame: pd.DataFrame, label: str, dates: list[str], key: list[str], prices: bool = True) -> pd.DataFrame:
    out = frame.copy()
    for column in dates:
        out[column] = pd.to_datetime(out[column], errors="coerce")
        if column != "datetime":
            out[column] = out[column].dt.normalize()
    for column in ("contract", "main_contract"):
        if column in out:
            out[column] = out[column].astype(str).str.strip()
    numeric = [column for column in ["open", "high", "low", "close", "settle", "oi", "amount", "volume"] if column in out]
    out[numeric] = out[numeric].apply(pd.to_numeric, errors="coerce")
    if out[dates].isna().any().any():
        raise ValueError(f"{label} 存在无法解析的日期")
    if out.duplicated(key).any():
        raise ValueError(f"{label} 存在重复键: {key}")
    if prices and ((out["high"] < out[["open", "low", "close"]].max(axis=1)) | (out["low"] > out[["open", "high", "close"]].min(axis=1))).any():
        raise ValueError(f"{label} 存在非法 OHLC")
    for column in ["oi", "amount", "volume"]:
        if column in out and (out[column].isna() | out[column].lt(0)).any():
            raise ValueError(f"{label} 的 {column} 必须非负且非空")
    return out.sort_values(key).reset_index(drop=True)


def apply_adjustment(frame: pd.DataFrame, factor: pd.Series) -> pd.DataFrame:
    out = frame.copy()
    out["adj_factor"] = pd.to_numeric(factor, errors="coerce")  # 唯一可见的价格复权函数
    for column in ["open", "high", "low", "close", "settle"]:
        if column in out:
            out[f"{column}_adj"] = out[column] * out["adj_factor"]  # 仅价格列乘因子
    return out

## 读取并标准化原始输入

这里不猜测文件用途；每一个来源都由配置单元中的显式路径控制。


In [3]:
source_mode = DATA_SOURCE.strip().lower()
if source_mode not in {"excel", "duckdb"}:
    raise ValueError('DATA_SOURCE 只能是 "excel" 或 "duckdb"')

daily_required = ["contract", "trade_date", "open", "high", "low", "close", "oi", "amount", "volume"]
if ROLL_PRICE_COLUMN == "settle":
    daily_required.append("settle")  # 未设置 close 时不允许悄悄回退
minute_required = ["datetime", "trading_date", "contract", "open", "high", "low", "close", "oi", "amount", "volume"]

db_daily_main = None
db_mapping = None
if source_mode == "excel":
    raw_daily_source = canonicalize(RAW_DAILY_PATH, RAW_DAILY_MAP, "原始日线", daily_required)
    raw_minute_source = canonicalize(RAW_MINUTE_PATH, RAW_MINUTE_MAP, "原始分钟", minute_required)
else:
    raw_daily_db, db_daily_main, raw_minute_db, db_mapping = load_duckdb_sources(DUCKDB_PATH)
    raw_daily_source = canonicalize_frame(raw_daily_db, RAW_DAILY_MAP, "DuckDB 原始日线", daily_required)
    raw_minute_source = canonicalize_frame(raw_minute_db, RAW_MINUTE_MAP, "DuckDB 主连分钟", minute_required)
    db_daily_main = parse_and_validate(
        canonicalize_frame(db_daily_main, RAW_DAILY_MAP, "DuckDB 主连日线", daily_required),
        "DuckDB 主连日线", ["trade_date"], ["trade_date", "contract"]
    )

raw_daily = parse_and_validate(raw_daily_source, "原始日线", ["trade_date"], ["trade_date", "contract"])
raw_minute = parse_and_validate(raw_minute_source, "原始分钟", ["datetime", "trading_date"], ["datetime", "contract"])
if raw_minute["datetime"].dt.normalize().ne(raw_minute["trading_date"]).any():
    raise ValueError("分钟 datetime 的自然日必须等于 trading_date")  # 阻断跨日标记错误
calendar_path = EVENT_CALENDAR_PATH if EVENT_CALENDAR_PATH.is_file() else LEGACY_EVENT_CALENDAR_PATH
eco_calendar_df = parse_and_validate(canonicalize(calendar_path, EVENT_CALENDAR_MAP, "事件日历", ["event_date", "event_name", "event_type"]), "事件日历", ["event_date"], ["event_date", "event_name", "event_type"], prices=False)
print(f"数据源: {source_mode}; 日线原始行数: {len(raw_daily):,}; 分钟原始/主连行数: {len(raw_minute):,}; 事件数: {len(eco_calendar_df):,}")

DuckDB 原始日线剔除 237 行不完整辅助合约记录；这些记录不进入主连或换月价格计算。
DuckDB 主连日线剔除 1 个尚未完整的交易日: ['2026-07-14']
数据源: duckdb; 日线原始行数: 7,809; 分钟原始/主连行数: 472,794; 事件数: 1,208


## 主力映射与原始主连

手工日线必须使用映射。外部日线在未配置映射文件时，使用外部连续日线自身的 `contract/main_contract` 作为隐式映射。


In [4]:
def read_mapping() -> pd.DataFrame:
    if source_mode == "duckdb":
        mapping = canonicalize_frame(db_mapping, MAIN_MAPPING_MAP, "DuckDB 主力映射", ["trade_date", "main_contract"])
    else:
        mapping = canonicalize(MAIN_MAPPING_PATH, MAIN_MAPPING_MAP, "主力映射", ["trade_date", "main_contract"])
    mapping = parse_and_validate(mapping, "主力映射", ["trade_date"], ["trade_date"], prices=False)
    if mapping.duplicated("trade_date").any():
        raise ValueError("主力映射每个交易日只能有一个主力合约")
    return mapping


external_daily = None
if MANUAL_ADJUST_DAILY:
    daily_mapping = read_mapping()  # 手工复权的合约切换由用户映射决定
else:
    external_daily = parse_and_validate(canonicalize(EXTERNAL_DAILY_PATH, EXTERNAL_DAILY_MAP, "外部复权日线", daily_required), "外部复权日线", ["trade_date"], ["trade_date", "contract"])
    if source_mode == "duckdb":
        daily_mapping = read_mapping()
    elif MAIN_MAPPING_PATH.is_file():
        daily_mapping = read_mapping()  # 用户若提供映射，优先保留其人工选择
    else:
        daily_mapping = external_daily[["trade_date", "contract"]].rename(columns={"contract": "main_contract"}).drop_duplicates("trade_date")  # 外部日线合约即隐式映射

if source_mode == "duckdb":
    available_main = db_daily_main[["trade_date", "contract"]].drop_duplicates()
    aligned_mapping = daily_mapping.merge(
        available_main,
        left_on=["trade_date", "main_contract"],
        right_on=["trade_date", "contract"],
        how="inner",
    )
    omitted = daily_mapping.loc[~daily_mapping["trade_date"].isin(aligned_mapping["trade_date"]), "trade_date"]
    if not omitted.empty:
        omitted_text = pd.to_datetime(omitted).dt.strftime("%Y-%m-%d").tolist()
        print(f"DuckDB 主力映射剔除 {len(omitted_text)} 个无完整日线的交易日: {omitted_text}")
    daily_mapping = aligned_mapping[["trade_date", "main_contract"]].sort_values("trade_date").reset_index(drop=True)
    daily_raw_main_df = db_daily_main.merge(
        daily_mapping,
        left_on=["trade_date", "contract"],
        right_on=["trade_date", "main_contract"],
        how="inner",
    ).drop(columns="main_contract")
else:
    daily_raw_main_df = raw_daily.merge(daily_mapping, left_on=["trade_date", "contract"], right_on=["trade_date", "main_contract"], how="inner").drop(columns="main_contract")
daily_raw_main_df = daily_raw_main_df.sort_values("trade_date").reset_index(drop=True)
if len(daily_raw_main_df) != len(daily_mapping):
    raise ValueError("日线主力映射无法在原始日线中找到对应合约")
daily_raw_main_df["ts_code"] = daily_raw_main_df["contract"]  # 下游流程兼容合约名

DuckDB 主力映射剔除 1 个无完整日线的交易日: ['2026-07-14']


## 日线：手工比例后复权或外部复权核验

手工模式严格复刻 `PriceAdj.py`：换月日 D 使用前一交易日 D-1 的新/旧合约价格，因子向历史累乘，最新日保持 1。


In [5]:
def manual_daily_factors(raw: pd.DataFrame, mapping: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    series = mapping.sort_values("trade_date").reset_index(drop=True).copy()
    roll_factor = pd.Series(1.0, index=series.index)
    prices = raw.set_index(["trade_date", "contract"])[ROLL_PRICE_COLUMN]
    warnings: list[dict[str, str | float | pd.Timestamp]] = []
    for index in range(1, len(series)):
        old_contract = series.at[index - 1, "main_contract"]
        new_contract = series.at[index, "main_contract"]
        if old_contract != new_contract:
            prior_date = series.at[index - 1, "trade_date"]  # D-1 是上一真实交易日
            old_price = prices.get((prior_date, old_contract))
            new_price = prices.get((prior_date, new_contract))
            reason = ""
            if old_price is None or pd.isna(old_price):
                reason = f"旧合约 D-1 {ROLL_PRICE_COLUMN} 缺失"
            elif new_price is None or pd.isna(new_price):
                reason = f"新合约 D-1 {ROLL_PRICE_COLUMN} 缺失"
            elif old_price == 0:
                reason = f"旧合约 D-1 {ROLL_PRICE_COLUMN} 为 0"
            if reason:
                roll_factor.at[index] = 1.0  # 与 PriceAdj.py 一致：不可计算时不调整
                warnings.append({"换月日": series.at[index, "trade_date"], "旧合约": old_contract, "新合约": new_contract, "前一交易日": prior_date, "原因": reason, "应用比例": 1.0})
            else:
                roll_factor.at[index] = new_price / old_price  # 新合约 / 旧合约，与 PriceAdj 同方向
    factor = pd.Series(1.0, index=series.index)
    for index in range(len(series) - 2, -1, -1):
        factor.at[index] = factor.at[index + 1] * roll_factor.at[index + 1]  # 历史日乘随后换月比例
    return factor, pd.DataFrame(warnings, columns=["换月日", "旧合约", "新合约", "前一交易日", "原因", "应用比例"])


if MANUAL_ADJUST_DAILY:
    daily_adj_factor_series, roll_warning_audit_df = manual_daily_factors(raw_daily, daily_mapping)
    daily_adjusted_df = apply_adjustment(daily_raw_main_df, daily_adj_factor_series)
else:
    external_main = external_daily.merge(daily_mapping, left_on=["trade_date", "contract"], right_on=["trade_date", "main_contract"], how="inner", suffixes=("", "_map"))
    aligned = daily_raw_main_df.merge(external_main[["trade_date", "contract", "open", "high", "low", "close", "settle"]], on=["trade_date", "contract"], suffixes=("_raw", "_external"))
    daily_adj_factor_series = aligned["close_external"] / aligned["close_raw"]  # 外部 close/raw close 推导真实因子
    if daily_adj_factor_series.isna().any() or daily_adj_factor_series.le(0).any():
        raise ValueError("外部日线无法推导正的 adj_factor")
    for column in ["open", "high", "low", "close"]:
        if not (aligned[f"{column}_raw"] * daily_adj_factor_series).sub(aligned[f"{column}_external"]).abs().le(1e-8).all():
            raise ValueError(f"外部日线 {column} 与推导因子不一致")
    daily_adjusted_df = apply_adjustment(daily_raw_main_df, daily_adj_factor_series)
    roll_warning_audit_df = pd.DataFrame(columns=["换月日", "旧合约", "新合约", "前一交易日", "原因", "应用比例"])
if daily_adjusted_df["adj_factor"].isna().any() or daily_adjusted_df["adj_factor"].le(0).any():
    raise ValueError("日线 adj_factor 必须非空且为正")
if not roll_warning_audit_df.empty:
    print("换月复权警告审计表：以下 D-1 价格不可计算，已按 PriceAdj.py 使用比例 1.0。")
    display(roll_warning_audit_df)  # 让人工复核每次显式回退

## 分钟：按日线主连筛选

分钟的 OI、成交额和成交量永远保持原值；只有 OHLC（以及存在时 settle）乘日线或外部验证后的复权因子。


In [6]:
minute_mapping = daily_mapping.rename(columns={"trade_date": "trading_date", "main_contract": "contract"})
if source_mode == "duckdb":
    # main_minute_continuous 已由 TreasuryFutures.ipynb 统一 Wind 与历史 CSV；不再次按映射删行。
    minute_raw_main_df = raw_minute[raw_minute["trading_date"].isin(daily_mapping["trade_date"])].copy()
else:
    minute_raw_main_df = raw_minute.merge(minute_mapping, on=["trading_date", "contract"], how="inner")
if minute_raw_main_df.empty:
    raise ValueError("原始分钟线没有任何日线主力合约记录")
minute_raw_main_df = minute_raw_main_df.sort_values(["trading_date", "datetime", "contract"]).reset_index(drop=True)
minute_raw_main_df["ts_code"] = minute_raw_main_df["contract"]  # 下游流程兼容合约名
daily_factor_by_date = daily_adjusted_df[["trade_date", "adj_factor"]].rename(columns={"trade_date": "trading_date"})
if MANUAL_ADJUST_MINUTE:
    minute_adjusted_df = minute_raw_main_df.merge(daily_factor_by_date, on="trading_date", how="left")
    minute_adjusted_df = apply_adjustment(minute_adjusted_df.drop(columns="adj_factor"), minute_adjusted_df["adj_factor"])
else:
    external_minute = parse_and_validate(canonicalize(EXTERNAL_MINUTE_PATH, EXTERNAL_MINUTE_MAP, "外部复权分钟", ["datetime", "trading_date", "contract", "open", "high", "low", "close", "oi", "amount", "volume"]), "外部复权分钟", ["datetime", "trading_date"], ["datetime", "contract"])
    minute_aligned = minute_raw_main_df.merge(external_minute, on=["datetime", "trading_date", "contract"], suffixes=("_raw", "_external"))
    if len(minute_aligned) != len(minute_raw_main_df):
        raise ValueError("外部复权分钟与原始主连分钟无法完整对齐")
    minute_factor = minute_aligned["close_external"] / minute_aligned["close_raw"]  # 使用外部价格推导，不伪造 1 因子
    if minute_factor.isna().any() or minute_factor.le(0).any():
        raise ValueError("外部分钟无法推导正的 adj_factor")
    for column in ["open", "high", "low", "close"]:
        if not (minute_aligned[f"{column}_raw"] * minute_factor).sub(minute_aligned[f"{column}_external"]).abs().le(1e-8).all():
            raise ValueError(f"外部分钟 {column} 与推导因子不一致")
    minute_adjusted_df = apply_adjustment(minute_raw_main_df, minute_factor)
if minute_adjusted_df["adj_factor"].isna().any() or minute_adjusted_df["adj_factor"].le(0).any():
    raise ValueError("分钟 adj_factor 必须非空且为正")

## 最终质量闸门

最终表保留 `contract` 供人工核对，同时增加 `ts_code` 与 `*_adj`，保证下游信号和回测无需改动。


In [7]:
def final_check(daily: pd.DataFrame, minute: pd.DataFrame) -> None:
    if daily.duplicated(["trade_date", "ts_code"]).any() or minute.duplicated(["datetime", "ts_code"]).any():
        raise ValueError("最终主连数据存在重复主键")
    if not set(minute["trading_date"]).issubset(set(daily["trade_date"])):
        raise ValueError("分钟交易日必须包含在日线交易日中")
    for frame, label in [(daily, "日线"), (minute, "分钟")]:
        required = ["ts_code", "open_adj", "high_adj", "low_adj", "close_adj", "adj_factor", "amount", "volume", "oi"]
        if frame[required].isna().any().any():
            raise ValueError(f"{label} 最终兼容字段不得为空")


final_check(daily_adjusted_df, minute_adjusted_df)

## 保存五张表与兼容输出

五个命名变量都仍在内存中；`validated_*` 是下游信号和回测的稳定输入。


In [8]:
WORKING_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "daily_raw_main.parquet": daily_raw_main_df,
    "daily_adjusted.parquet": daily_adjusted_df,
    "minute_raw_main.parquet": minute_raw_main_df,
    "minute_adjusted.parquet": minute_adjusted_df,
    "eco_calendar.parquet": eco_calendar_df,
    "validated_daily.parquet": daily_adjusted_df,
    "validated_minute.parquet": minute_adjusted_df,
    "validated_events.parquet": eco_calendar_df,
}
for filename, frame in outputs.items():
    frame.to_parquet(WORKING_DIR / filename, index=False)  # 显式保存，便于导师逐表复核
print(f"已从 {source_mode} 数据源保存五张主表和下游兼容输出到: {WORKING_DIR}")

已从 duckdb 数据源保存五张主表和下游兼容输出到: E:\CodeBase\TaiKangAM\TreasuryFutures\Factor\notebooks\CICC_CloseSession_Reverse\CTreasuryFutures\factors\cicc_close_session_reverse\working
